# 01 - Prepare the RoadWatcher Data
This notebook sets up the road vehicle images dataset, inspects folders, cleans invalid or corrupt files, creates the reproducible test split, and writes the dataset YAML config.

In [ ]:
# 1. Imports
import os
import shutil
import random
import hashlib
from pathlib import Path
from collections import Counter
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import yaml

In [ ]:
# 2. Check GPU and PyTorch
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

In [ ]:
# 3. Project paths
project_dir = Path.cwd().parent
src_dir = project_dir / "trafic_data"
data_dir = project_dir / "data_clean"

print("Project folder: ", project_dir)
print("Source raw data:", src_dir)
print("Clean dataset:  ", data_dir)

In [ ]:
# 4. Inspect source dataset folders
print("Source directory contents:")
for item in os.listdir(src_dir):
    item_path = src_dir / item
    if item_path.is_dir():
        print("Folder:", item + "/")
    else:
        print("File:  ", item)

In [ ]:
# 5. Count images and labels in raw dataset
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for split in ["train", "valid"]:
    imgs = [f for f in (src_dir / split / "images").iterdir() if f.is_file() and f.suffix.lower() in extensions]
    lbls = list((src_dir / split / "labels").glob("*.txt"))
    print(split.upper(), "-> Images:", len(imgs), ", Labels:", len(lbls))

In [ ]:
# 6. Copy raw data to data_clean/
# We make a working copy so the original raw data is never modified
if not data_dir.exists():
    shutil.copytree(src_dir, data_dir, ignore=shutil.ignore_patterns("*.cache"))
    print("Copied raw data to:", data_dir)
else:
    print("data_clean/ already exists - skipping copy")

In [ ]:
# 7. Check for empty labels and corrupt images, and clean them
for split in ["train", "valid"]:
    images_folder = data_dir / split / "images"
    labels_folder = data_dir / split / "labels"
    
    deleted_empty = 0
    deleted_corrupt = 0
    
    for img_file in list(images_folder.iterdir()):
        if not img_file.is_file() or img_file.suffix.lower() not in extensions:
            continue
            
        lbl_file = labels_folder / (img_file.stem + ".txt")
        
        # Check if label exists and is not empty
        if not lbl_file.exists() or lbl_file.read_text().strip() == "":
            img_file.unlink(missing_ok=True)
            if lbl_file.exists():
                lbl_file.unlink(missing_ok=True)
            deleted_empty += 1
            continue
            
        # Check if image file can be opened
        try:
            with Image.open(img_file) as img:
                img.verify()
        except Exception:
            img_file.unlink(missing_ok=True)
            lbl_file.unlink(missing_ok=True)
            deleted_corrupt += 1

    print(split, ": deleted", deleted_empty, "empty/missing labels and", deleted_corrupt, "corrupt images")

In [ ]:
# 8. Check image and label matching
for split in ["train", "valid"]:
    images = {p.stem for p in (data_dir / split / "images").iterdir() if p.is_file() and p.suffix.lower() in extensions}
    labels = {p.stem for p in (data_dir / split / "labels").glob("*.txt")}
    
    missing_labels = images - labels
    missing_images = labels - images
    
    # Remove any unmatched orphans
    for stem in missing_labels:
        for f in (data_dir / split / "images").glob(stem + ".*"):
            f.unlink(missing_ok=True)
    for stem in missing_images:
        (data_dir / split / "labels" / (stem + ".txt")).unlink(missing_ok=True)
        
    print(split.upper(), ": images =", len(images), ", labels =", len(labels), ", perfect match =", images == labels)

In [ ]:
# 9. Create reproducible test split (405 images from train using seed 42)
test_images = data_dir / "test" / "images"
test_labels = data_dir / "test" / "labels"

test_images.mkdir(parents=True, exist_ok=True)
test_labels.mkdir(parents=True, exist_ok=True)

existing_test = list(test_images.iterdir())

if len(existing_test) == 0:
    train_imgs = sorted([p for p in (data_dir / "train" / "images").iterdir() if p.suffix.lower() in extensions])
    random.seed(42)
    random.shuffle(train_imgs)
    
    selected_for_test = train_imgs[:405]
    
    for img in selected_for_test:
        lbl = data_dir / "train" / "labels" / (img.stem + ".txt")
        shutil.move(str(img), str(test_images / img.name))
        if lbl.exists():
            shutil.move(str(lbl), str(test_labels / lbl.name))
            
    print("Moved", len(selected_for_test), "images to test set")
else:
    print("Test set already exists with", len(existing_test), "images")

In [ ]:
# 10. Verify final split counts (Brief target: 2297 train, 298 valid, 405 test)
expected_counts = {"train": 2297, "valid": 298, "test": 405}

for split in ["train", "valid", "test"]:
    img_count = sum(1 for f in (data_dir / split / "images").iterdir() if f.is_file() and f.suffix.lower() in extensions)
    lbl_count = len(list((data_dir / split / "labels").glob("*.txt")))
    print(split.upper(), "-> Images:", img_count, ", Labels:", lbl_count, "(Expected:", str(expected_counts[split]) + ")")

In [ ]:
# 11. Check for exact duplicate images using MD5 hash
def image_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

seen_hashes = {}
duplicate_count = 0

for split in ["train", "valid", "test"]:
    images_dir = data_dir / split / "images"
    for img in sorted(images_dir.iterdir()):
        if img.suffix.lower() not in extensions:
            continue
        h = image_hash(img)
        if h in seen_hashes:
            print("Duplicate found:", split + "/" + img.name, "is identical to", seen_hashes[h])
            duplicate_count += 1
        else:
            seen_hashes[h] = split + "/" + img.name

print("Total exact duplicates found across splits:", duplicate_count)

In [ ]:
# 12. Read class names and create roadwatcher.yaml
with open(src_dir / "data_1.yaml", "r") as f:
    config = yaml.safe_load(f)

class_names = config["names"]
nc = int(config["nc"])

print("Total classes (" + str(nc) + "):", class_names)

# Create dataset yaml required by YOLOv9
roadwatcher_config = {
    "path": str(data_dir.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": nc,
    "names": class_names
}

yaml_path = project_dir / "roadwatcher.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(roadwatcher_config, f, sort_keys=False)

print()
print("Saved roadwatcher.yaml successfully:")
print(yaml_path.read_text())

In [ ]:
# 13. Count objects per class and plot distribution
for split in ["train", "valid", "test"]:
    labels_dir = data_dir / split / "labels"
    class_counts = Counter()
    
    for lbl_file in labels_dir.glob("*.txt"):
        with open(lbl_file, "r") as f:
            for line in f:
                parts = line.split()
                if len(parts) == 5:
                    class_counts[int(parts[0])] += 1
                    
    print()
    print("---", split.upper(), "OBJECT COUNTS ---")
    for cid in range(nc):
        print("Class", cid, "(" + class_names[cid] + "):", class_counts[cid], "objects")
        
    # Plot bar chart for this split
    plt.figure(figsize=(10, 3.5))
    plt.bar([class_names[i] for i in range(nc)], [class_counts[i] for i in range(nc)], color="steelblue")
    plt.xticks(rotation=75, ha="right")
    plt.title("Class Distribution - " + split.upper())
    plt.ylabel("Objects")
    plt.tight_layout()
    plt.show()

In [ ]:
# 14. Visualize 6 sample images with bounding boxes
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
sample_images = sorted([p for p in (data_dir / "train" / "images").iterdir() if p.suffix.lower() in extensions])[:6]

for ax, img_path in zip(axes.flat, sample_images):
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = img.size
    
    lbl_path = data_dir / "train" / "labels" / (img_path.stem + ".txt")
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if len(parts) == 5:
                cls_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:])
                x1 = (xc - bw / 2) * w
                y1 = (yc - bh / 2) * h
                x2 = (xc + bw / 2) * w
                y2 = (yc + bh / 2) * h
                draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
                draw.text((x1, y1), class_names[cls_id], fill="yellow")
                
    ax.imshow(img)
    ax.axis("off")

plt.suptitle("Sample Training Images with Bounding Boxes", fontsize=14)
plt.tight_layout()
plt.show()